# Baseline Repair — ReferenceResidualUNetV2 (s2repair)

Repairs the audited v1 baseline into a **physically-bounded** model and validates it
through three gates. Never modifies `best.pt` or the dataset.

**Attach (Add Input):** (1) synthetic dataset, (2) training output with `best.pt`,
(3) *optionally* the audit output with `ground_truth_filter_manifest.json`.

In [ ]:
REPO_URL = "https://github.com/Yuv1ne04/trial3kaggle.git"
import os
!rm -rf /kaggle/working/trial3
!git clone -q $REPO_URL /kaggle/working/trial3
%cd /kaggle/working/trial3
!git log --oneline -1
!pip -q install rasterio pyyaml >/dev/null 2>&1
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# Locate dataset, checkpoint, and (optional) audit manifest under /kaggle/input
import os, glob
PRUNE={"patch_library","cloud_tile_library","mask_library","reference_library","target_library",
       "samples","train","validation","test",".git"}
def find_root(base="/kaggle/input"):
    for dp,dn,fn in os.walk(base):
        if "samples" in dn and (os.path.isdir(os.path.join(dp,"samples","test")) or os.path.isdir(os.path.join(dp,"samples","train"))):
            return dp
        dn[:]=[d for d in dn if d not in PRUNE]
def find_file(name, bases=("/kaggle/input","/kaggle/working")):
    for base in bases:
        for dp,dn,fn in os.walk(base):
            if name in fn: return os.path.join(dp,name)
            dn[:]=[d for d in dn if d not in PRUNE]
DATA=find_root(); CKPT=find_file("best.pt"); MANIFEST=find_file("ground_truth_filter_manifest.json")
print("DATA:", DATA)
print("CKPT:", CKPT)
print("MANIFEST:", MANIFEST)
assert DATA and CKPT

## Part 1 — bounded-output diagnostic (best.pt unchanged)

In [ ]:
!python -m s2repair diagnose --checkpoint "$CKPT" --dataset "$DATA" --output /kaggle/working/repair     --max-samples 0 --batch-size 32 --device auto

## Part 2 — worst-case failure analysis

In [ ]:
!python -m s2repair worstcase --checkpoint "$CKPT" --dataset "$DATA" --output /kaggle/working/repair     --max-samples 3000 --top-k 50 --render 20 --batch-size 32 --device auto

## Part 4 — reference-input capability

In [ ]:
!python -m s2repair capability --dataset "$DATA" --output /kaggle/working/repair

## Gate 1 — tiny overfit (must PASS before Gate 2)
Proves the bounded model fits and beats the weighted-reference mean on a tiny high-quality subset.

In [ ]:
M = f'--audit-manifest "{MANIFEST}"' if MANIFEST else ''
!python -m s2repair gate1 --config configs/reference_unet_v2_gate1.yaml     --dataset "$DATA" --output /kaggle/working/repair $M --device auto
import json; print(json.load(open('/kaggle/working/repair/gate1/gate1_report.json'))['status'])

## Gate 2 — small pilot (run only if Gate 1 PASSED)
Must beat the weighted-reference mean on a held-out validation subset.

In [ ]:
!python -m s2repair gate2 --config configs/reference_unet_v2_gate2.yaml     --dataset "$DATA" --output /kaggle/working/repair $M --device auto

## Gate 3 — 30k scale (run only if Gate 2 PASSED)
Same scale as the original checkpoint, for a fair comparison. ~10-12 h on a T4 —
use Save Version → Save & Run All (Commit).

In [ ]:
!python -m s2repair gate3 --config configs/reference_unet_v2_gate3.yaml     --dataset "$DATA" --output /kaggle/working/repair $M --device auto

## Outputs (`/kaggle/working/repair/`)
- `current_checkpoint_bounding_comparison.csv`, `prediction_range_by_band.csv`
- `worst_case_samples.csv`, `worst_case_panels/`
- `reference_input_capability_report.json`
- `gate1/gate1_report.json`, `gate2/gate2_report.json`, `gate3/…` + `checkpoints/best_rmse.pt`

Each gate refuses to auto-continue after a failure — inspect the report before proceeding.